## Emergency Medicine Triage RAG System

This notebook implements a complete Retrieval-Augmented Generation (RAG) system for emergency medicine triage decision support. The system processes new patient cases by:
1. Analyzing the case description
2. Retrieving relevant triage guidelines 
3. Finding similar past cases
4. Generating a comprehensive triage assessment

****RAG implementation with BOTH Specific Guideline sections and similar Past Cases**

In [ ]:
import json
import re
from typing import List, Dict, Any, Optional, Tuple, TypedDict, Literal
import os
from dotenv import load_dotenv
from tqdm import tqdm
from tqdm.notebook import tqdm
import traceback
import time

# Pydantic imports for structured outputs
from pydantic import BaseModel, Field, ValidationError

# LangChain imports
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# LangGraph imports
from langgraph.graph import StateGraph, END

## System Architecture Overview

This notebook implements a multi-stage RAG pipeline using a graph-based architecture. The key components are:

1. **Data Sources**:
   - Triage guidelines structured in sections
   - Past emergency cases with known triage outcomes
   - Vector embeddings of past cases for similarity search

2. **LLM setup**:
   - **LLM Initialization**(choose one): Used to generate clinical summaries and reasoning task
        1. **DeepSeek-v3** (updated to deepseek-v4-pro)
        2. **gpt-4o**
        3. **claude-3-7-sonnet**
   - Setting up Azure OpenAI embeddings for vector search

3. **Pipeline Flow**:
   - Case preprocessing and summarization
   - Guideline retrieval and selection
   - Specialty prediction
   - Similar case retrieval
   - Integrated final assessment
   
Each component is implemented as a node in a LangGraph that manages state transitions through the pipeline.

## Step 1: Initial Setup and Configuration

This section initializes all required components:

1. **Environment Configuration**:
   - Loading API keys and endpoints from environment variables
   
2. **LLM setup**:
   - **LLM Initialization**(choose one): Used to generate clinical summaries and reasoning task
        1. **DeepSeek-v3**
        2. **gpt-4o**
        3. **claude-3-7-sonnet**
        4. **kimi-k2.5**
   - Setting up Azure OpenAI embeddings for vector search

3. **Data Loading**:
   - Triage guidelines with section summaries
   - Vector database of past cases
   - Specialty mapping for case categorization
   
4. **State Definition**:
   - Defining the `GraphState` class to track data through the pipeline
   - Managing transitions between processing stages

In [ ]:
# Load environment variables
load_dotenv()

# Verify environment variables
required_env_vars = [
    "AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_API_VERSION",
    "DEEPSEEK_API_KEY",
]

missing_vars = [var for var in required_env_vars if not os.getenv(var)]
if missing_vars:
    raise ValueError(f"Missing environment variables: {', '.join(missing_vars)}")

print("Environment variables loaded successfully.")

In [ ]:
from langchain_deepseek import ChatDeepSeek

# Initialize LLM
llm = ChatDeepSeek(
    model="deepseek-v4-pro",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    temperature=0,
    # Move extra_body here directly:
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }
)
print("LLM initialized successfully.")

model = "deepseek-v4-pro"

In [ ]:
# from langchain_openai import AzureChatOpenAI


# def initialize_azure_openai(
#     azure_deployment_name: str = os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME"),
#     azure_endpoint: str = os.getenv("AZURE_OPENAI_ENDPOINT"),
#     azure_api_key: str = os.getenv("AZURE_OPENAI_API_KEY"),
#     azure_api_version: str = os.getenv("AZURE_OPENAI_API_VERSION"),
#     temperature: float = 0,
# ):
#     """
#     Initialize Azure OpenAI client

#     Args:
#         azure_deployment_name (str): Azure deployment name
#         azure_endpoint (str): Azure endpoint URL
#         azure_api_key (str): Azure API key
#         azure_api_version (str): Azure API version
#         temperature (float): Temperature for generation

#     Returns:
#         AzureChatOpenAI: Initialized Azure OpenAI client
#     """
#     return AzureChatOpenAI(
#         azure_deployment=azure_deployment_name,
#         azure_endpoint=azure_endpoint,
#         api_key=azure_api_key,
#         api_version=azure_api_version,
#         temperature=temperature,
#     )


# llm = initialize_azure_openai()

# print("LLM initialized successfully.")

# model = "gpt-4o"

In [ ]:
# from langchain_anthropic import ChatAnthropic


# def initialize_claude(
#     api_key: str = os.getenv("ANTHROPIC_API_KEY"),
#     model_name: str = os.getenv("ANTHROPIC_MODEL_NAME"),
#     temperature: float = 0,
# ):
#     """

#     Initialize Claude client using LangChain


#     Args:
#         api_key (str): Anthropic API key
#         model_name (str): Model name to use (claude-3-7-sonnet-20250219 for latest Claude 3.7 Sonnet)
#         temperature (float): Temperature for generation

#     Returns:
#         ChatAnthropic: Initialized Claude client
#     """

#     return ChatAnthropic(
#         model=model_name,
#         anthropic_api_key=api_key,
#         temperature=temperature
#     )


# llm = initialize_claude()

# print("Claude LLM initialized successfully.")

# model = os.getenv("ANTHROPIC_MODEL_NAME")
# print(model)

In [ ]:
def initialize_embeddings() -> AzureOpenAIEmbeddings:
    """Initialize Azure OpenAI embeddings"""
    return AzureOpenAIEmbeddings(
        azure_deployment=os.getenv("AZURE_OPENAI_EMBEDDINGS_DEPLOYMENT_NAME"),
        azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
        api_key=os.getenv("AZURE_OPENAI_API_KEY"),
        api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    )


# Initialize embeddings
embeddings = initialize_embeddings()
print("Embeddings initialized successfully.")

In [ ]:
# Load the triage guidelines from the JSON file
def load_triage_guidelines(file_path) -> List[Dict[str, str]]:
    """
    Load the triage guidelines from a JSON file.
    
    Args:
        file_path (str): Path to the JSON file
        
    Returns:
        List[Dict[str, str]]: List of guideline sections with title, content, and summary
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        guidelines = json.load(f)
    
    print(f"Loaded {len(guidelines)} sections from {file_path}")
    return guidelines

# Load the guidelines
triage_guidelines = load_triage_guidelines(file_path=f"../../db/triage_sections_with_summaries_{model}.json")

# Extract just the metadata (title and summary) for retrieval decisions
guideline_section_metadata = [
    {"section_title": section["section_title"], "summary": section["summary"]}
    for section in triage_guidelines
]

In [ ]:
# Load the Vector DB (Past Cases)

# Constants and configurations
BASE_DIR = f"../../db/past_case/case_vectordb_{model}_3000case"

# Load existing vector databases
def load_vector_databases(base_dir: str) -> Chroma:
    """
    Load existing Chroma vector databases
    
    Args:
        base_dir: Base directory containing the vector databases
        
    Returns:
        Chroma VectorDB
    """
    summary_vectordb = Chroma(
        persist_directory=os.path.join(base_dir, "summary_vectordb"),
        embedding_function=embeddings,
        collection_name="summary_text"
    )
    
    return summary_vectordb

# Load vector database
summary_vectordb = load_vector_databases(BASE_DIR)

# Print a sample of IDs
print(f"Summary DB: {summary_vectordb.get()['ids'][:3]}...")

# Print the number of documents in each database
print(f"Summary DB document count: {summary_vectordb._collection.count()}")
print("Vector databases loaded successfully.")

# set past case full json dir 
case_json_dir = os.path.join(BASE_DIR, "json_store")
print(f"Past Case Json Store dir: {case_json_dir}")

k_cases = 5
print(f"will retrieve top-{k_cases} past-cases for each case")

In [ ]:
# Load unique values of "Attending Specialty" from vector database metadata
all_metadatas = summary_vectordb.get()["metadatas"]
unique_specialties = list({
    m["Attending Specialty"]
    for m in all_metadatas
    if "Attending Specialty" in m and m["Attending Specialty"]
})
print(unique_specialties)

In [ ]:
# Perform Mapping
AGENT_SPECIALTY_LIST = [
    "Medicine",
    "Surgery",
    "Obstetrics and Gynaecology",
    "Oncology",
    "Neurosurgery",
    "Orthopaedics",
    "Paediatrics",
    "Eye",
    "Psychiatry",
    "ENT",
    "Dermatology",
    "Dental",
    "Others",
]

# Mapping from standardized names to the actual values in the dataset
SPECIALTY_MAPPING = {
    "Medicine": ["Medicine", "IDU"],
    "Surgery": ["Surg", "Surgery", "Burn", "Others: UROLOGY", "Others: URO", "CTS"],
    "Obstetrics and Gynaecology": ["Gynae", "Obs/Mat"],
    "Oncology": [
        "Others: Onco",
        "Others: Oncology",
        "Others: onco",
        "Others: oncology",
        "RT",
        "Others: ONCO",
        "Others: ONC",
    ],
    "Neurosurgery": ["NS"],
    "Orthopaedics": ["O & T", "Orthopaedics"],
    "Paediatrics": ["Paed"],
    "Eye": ["Eye"],
    "Psychiatry": ["Psych"],
    "ENT": ["ENT"],
    "Dermatology": ["Derma"],
    "Dental": ["Dental"],
    "Others": ["Others"],
}

# Reverse mapping for lookup (actual values -> standardized names)
REVERSE_SPECIALTY_MAPPING = {}
for standard, variations in SPECIALTY_MAPPING.items():
    for variation in variations:
        REVERSE_SPECIALTY_MAPPING[variation] = standard


def map_specialty_to_dataset_values(specialty: str) -> List[str]:
    """
    Map a standardized specialty name to all its variations in the dataset

    Args:
        specialty: Standardized specialty name

    Returns:
        List of corresponding values in the dataset
    """
    if not specialty:
        return []

    return SPECIALTY_MAPPING.get(specialty, [])


def get_standardized_specialty(dataset_value: str) -> str:
    """
    Map a value from the dataset to its standardized specialty name

    Args:
        dataset_value: Value from the dataset

    Returns:
        Standardized specialty name
    """
    return REVERSE_SPECIALTY_MAPPING.get(dataset_value, "Others")

In [ ]:
# Define state structure
class GraphState(dict):
    """State for the retrieval graph."""

    summary_text: Optional[str] = (
        None                                                           # clinical summary of new case 
    )
    metadata_filter: Dict[str, Any]                                    # filter for past case retrieval
    case_json_full: Dict[str, Any]                                     # full case json
    guideline_section_metadata: List[
        Dict[str, str]
    ]                                                                  # guideline section metadata: title, summary
    selected_guideline_sections: Optional[List[str]] = None            # selected guideline sections
    retrieved_guideline_content: Optional[List[Dict[str, str]]] = (
        None                                                           # selected guideline sections content
    )
    specialty_list: List[str]                                          # standardized specialty list
    selected_attending_specialty: Optional[Dict[str, Any]] = None      # selected specialty
    retrieved_cases: List[Dict[str, Any]] = []                         # retrieved past cases
    retrieved_cases_content: List[Dict[str, Any]] = []                 # retrieved past cases content
    past_cases_context: str = ""                                      
    final_assessment: Optional[str] = None                             # final assessment output    guideline_reasoning: Optional[str] = None                       # reasoning for guideline selection
    final_category: Optional[str] = None                           # final triage category
    final_confidence: Optional[str] = None                         # final confidence level
    processing_failed: Optional[bool] = False                      # whether processing failed
    failure_reason: Optional[str] = None                           # reason for failure
    failed_node: Optional[str] = None                              # node where failure occurred


In [ ]:
# Pydantic schemas for structured outputs
class GuidelineRetrievalOutput(BaseModel):
    """Structured output for guideline retrieval decision"""
    selected_sections: List[str] = Field(
        description="List of section_title(s) exactly as they appear in the guidelines, maximum 2",
        max_items=2
    )
    reasoning: str = Field(description="Brief explanation for section selection")

class TriageStep(BaseModel):
    """Individual step in triage prediction process"""
    category: Literal["1", "2", "3", "4", "5", "Not specified"] = Field(description="Triage category")
    confidence: Literal["High", "Medium", "Low"] = Field(description="Confidence level")
    reason: str = Field(description="Reasoning for this step")

class TriagePredictionOutput(BaseModel):
    """Structured output for final triage prediction"""
    step1_clinical_risk: TriageStep = Field(description="Clinical risk assessment")
    step2_guidelines: TriageStep = Field(description="Guidelines-based assessment")
    step3_realworld_factors: TriageStep = Field(description="Real-world factors adjustment")
    final_decision: TriageStep = Field(description="Final triage decision")

class TokenUsageMetrics(BaseModel):
    """Token usage metrics for LLM calls"""
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0

def create_specialty_prediction_schema(specialty_list: List[str]):
    """Dynamically create SpecialtyPredictionOutput using the provided specialty list."""
    SpecialtyLiteral = Literal[tuple(specialty_list)]

    class SpecialtyPredictionOutput(BaseModel):
        """Structured output for specialty prediction"""
        specialties: List[SpecialtyLiteral] = Field(
            description=(
                "Ordered list of 1-2 attending specialties from the allowed list — "
                "primary first, optional secondary second. "
                "Each value must be exactly one entry from the allowed list."
            ),
            min_items=1,
            max_items=2,
        )
        explanation: str = Field(
            description=(
                "Your clinical reasoning for the specialty selection, "
                "covering both primary and secondary choices (free text)"
            )
        )

    return SpecialtyPredictionOutput

## Step 2: Utility Functions

These utility functions handle data processing tasks throughout the pipeline:

1. **Case Loading**:
   - Functions to load and preprocess case JSON files
   
2. **Metadata Filtering**:
   - Functions to filter vector database results by metadata
   - Enables more targeted retrieval of similar past cases

In [ ]:
# Initial loading of JSON file (new case)
def load_case_from_json(file_path: str) -> Dict[str, Any]:
    """
    Load a FULL case from a JSON file as NEW case.

    Args:
        file_path (str): Path to the JSON file

    Returns:
        Dict[str, Any]: FULL Case data, with "Case Disposition" removed(required for test/val set).
    """
    with open(file_path, "r", encoding="utf-8") as f:
        case_data = json.load(f)

    # Remove sections if it exists, required for ground_truth set
    if "Case Disposition" in case_data:
        case_data.pop("Case Disposition")
    if "Ground Truth" in case_data:
        case_data.pop("Ground Truth")

    return case_data

In [ ]:
# filtering past case in vectordb according to metadata filters
def filter_documents_by_metadata(
    vectordb: Chroma, metadata_filters: Dict[str, Any], threshold: float = 1.0
) -> List[str]:
    """
    Filter documents by metadata criteria and return doc_ids that meet the criteria

    Args:
        vectordb: The vector database to search in
        metadata_filters: Dictionary of metadata filters to apply
        threshold: Minimum fraction of filters that must match (0.0-1.0)

    Returns:
        List of document IDs that match the metadata criteria
    """
    # Get all documents
    all_docs = vectordb.get()
    all_metadatas = all_docs["metadatas"]
    doc_ids = all_docs["ids"]

    matching_ids = []

    for i, metadata in enumerate(all_metadatas):
        # Count how many filters match
        match_count = 0
        total_applicable_filters = 0

        for key, value in metadata_filters.items():
            # Skip if the filter key doesn't exist in this document's metadata
            if key not in metadata:
                continue

            total_applicable_filters += 1

            # Check if values match (case-insensitive for string values)
            if isinstance(metadata[key], str) and isinstance(value, str):
                if metadata[key].lower() == value.lower():
                    match_count += 1
                    # print(f"Matched {key}: {metadata[key]} with {value}")
            else:
                if metadata[key] == value:
                    match_count += 1
                    # print(f"Matched {key}: {metadata[key]} with {value}")

        # Only include if at least threshold percentage of filters match
        if (
            total_applicable_filters > 0
            and (match_count / total_applicable_filters) >= threshold
        ):
            matching_ids.append(doc_ids[i])

    return matching_ids

## RAG Pipeline Implementation

The following sections implement the core RAG pipeline using a modular node-based approach. Each node performs a specific function and passes its results to subsequent nodes through the shared state object.

## Step 3: Node Definitions

Each node in the LangGraph represents a discrete processing step in the RAG pipeline. These nodes transform the input state and pass the results to downstream nodes.

### Core Pipeline Components

The pipeline consists of three main stages:
1. **Preprocessing**: Analyzing and summarizing the input case
2. **Retrieval**: Finding relevant guidelines and similar past cases 
3. **Generation**: Producing the final triage assessment

### 3.1: Preprocessing Node
The preprocessing node converts the raw case data into a concise clinical summary that captures the most relevant medical information. This summary serves as the query for both guideline retrieval and case similarity search, ensuring that retrieval focuses on clinically significant elements.

In [ ]:
# Define node to preprocess case data
def process_case_summary(state: GraphState, token_tracker=None) -> GraphState:

    ## Create Summary for VectorDB retrieval

    sections_to_expand = {}

    sections_for_summary = {}
    if "Demographics" in state["case_json_full"]:
        if "Age" in state["case_json_full"]["Demographics"]:
            sections_for_summary["Age"] = (
                state["case_json_full"].get("Demographics", {}).get("Age", "")
            )
        if "Sex" in state["case_json_full"]["Demographics"]:
            sections_for_summary["Sex"] = (
                state["case_json_full"].get("Demographics", {}).get("Sex", "")
            )
    if "Vitals and Observations" in state["case_json_full"]:
        sections_for_summary["Vitals and Observations"] = state["case_json_full"].get(
            "Vitals and Observations", {}
        )
    if "Clinical Presentation" in state["case_json_full"]:
        sections_for_summary["Clinical Presentation"] = state["case_json_full"].get(
            "Clinical Presentation", {}
        )

    summary_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
        You are a specialized medical summarization assistant for emergency triage notes. Your task is to analyze structured JSON data from triage notes and create a concise, retrievable summary highlighting only the most important clinical information.
        ## INPUT STRUCTURE
        You will receive structured JSON data containing:
        - Demographics (sex, age)
        - Clinical Presentation (chief complaint, condition on arrival)
        - Vitals and Observations

        ## OUTPUT REQUIREMENTS
        1. Generate a concise (2-3 sentence) summary focusing on:
           - Patient demographics (sex, age)
           - Key presenting complaints
           - Abnormal physical findings
           - ONLY abnormal vital signs according to strict criteria

        2. Write in a factual, medical style optimized for vector embedding retrieval.

        ## MEDICAL TERMINOLOGY GUIDELINES
        - Interpret medical abbreviations contextually 
        - Fix obvious typos/misspellings based on medical context
        - Maintain medical precision while expanding clarity
        - For Hong Kong hospital/facility abbreviations:
          - "PMH" → "Princess Margaret Hospital"
          - "KCH" → "Kwai Chung Hospital"
          - "OAH" → "Old Age Home"
          - If uncertain about any abbreviation, do NOT expand it

        ## VITAL SIGNS ABNORMALITY CRITERIA
        ONLY mention vital signs that meet these specific abnormality thresholds:

        - Blood Pressure (Adult):
          - Hypertension: SBP > 180 mmHg or DBP > 100 mmHg
          - Hypotension: SBP < 90 mmHg

        - Heart Rate (Adults):
          - Tachycardia: HR > 120/min
          - Bradycardia: HR < 50/min

        - Temperature (Adult):
          - Fever: Temp ≥ 38°C
          - Hypothermia: Temp < 35°C

        - Blood Sugar:
          - Hyperglycemia: H'stix > 16 mmol/l
          - Hypoglycemia: H'stix < 3.9 mmol/l

        - Respiratory Status (always consider):
          - Oxygen supplementation (if present)
          - SpO2 level if abnormal
          - Abnormal respiratory rate

        - Conscious Level:
          - GCS score if not 15/15
          - Pupil abnormalities (if provided)

        - Pregnancy/Menstruation: Note if pregnant or abnormal menstrual patterns

        ## EXAMPLES OF GOOD SUMMARIES:

        ### Example 1:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "F",
                    "Age": "56 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "R/F ? IHD, tachycardia oedema\\nSOB , LL edema and neck swelling x 10/7 \\nchest discomfort - , palpitiation",
                    "Condition on Arrival": "speak in sentence"
                }},
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "E: 4, V: 5, M: 6, Score: 15/15",
                        "BP": "1st: 144/89 mmHg",
                        "PR/AR": "99",
                        "Temp": "36.2",
                        "SpO2": "100% on room air, --",
                        "RR": "--",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }},
                    "Menstruation / Pregnancy Status": "unspecified"
                }}
            }}
        ]
        
        Output: "56-year-old female presenting with shortness of breath, lower limb edema, and neck swelling for 10 days, accompanied by chest discomfort and palpitations. Patient can speak in full sentences and has been referred for evaluation of possible ischemic heart disease with tachycardia and edema. All vital signs are within normal ranges."

        ### Example 2:
        Input: [
            {{
                "Demographics": {{
                    "Sex": "M",
                    "Age": "64 years"
                }},
                "Clinical Presentation": {{
                    "Chief complaint": "SOB this morning 0730\\nroom air 70%",
                    "Condition on Arrival": "no lower limb oedema\\nsputum sound+/-"
                }},    
                "Vitals and Observations": {{
                    "Vital Signs": {{
                        "GCS": "",
                        "BP": "1st: 165/97 mmHg",
                        "PR/AR": "130",
                        "Temp": "38.1",
                        "SpO2": "96% on 6 L/min via Nasal Cannula",
                        "RR": "20",
                        "PFR": "--",
                        "Limbs": "--",
                        "Pupil": "--"
                    }},
                    "Triage Intervention": {{
                        "H'stix": "",
                        "H'cue": ""
                    }}
                }}
            }}
        ]
        Output: "64-year-old male presenting with sudden onset shortness of breath at 7:30 AM with significant hypoxia (70% on room air). Patient has tachycardia (HR 130), fever (38.1°C), and requires oxygen supplementation (6 L/min via Nasal Cannula). Clinical examination reveals sputum sounds without lower limb edema."

        ## IMPORTANT REMINDERS:
        - Focus on clinical relevance and retrieval effectiveness
        - Maintain medical accuracy while being concise
        - Only highlight abnormal vitals based on specific thresholds
        - Fix obvious errors but maintain meaning integrity
        - Adjust abnormality thresholds for pediatric cases
        """,
            ),
            (
                "user",
                """
        Case Data (JSON):
        {case_data}

        OUTPUT (Summary):
        """,
            ),
        ]
    )

    # Prepare the input for the prompt
    case_data_str = json.dumps(sections_for_summary, indent=2)
    input_values = {"case_data": case_data_str}
    # Get the LLM response
    chain = summary_prompt | llm | StrOutputParser()
    _cb = {"callbacks": [token_tracker]} if token_tracker else {}
    response = chain.invoke(input_values, config=_cb)

    updated_state = state.copy()
    updated_state["summary_text"] = response
    return updated_state

### 3.2 Retrieval Nodes

The retrieval nodes perform three key functions:
1. **Guideline selection**: Identifying the most relevant guideline sections for the current case
2. **Specialty prediction**: Determining which medical specialty would typically handle this case
3. **Similar case retrieval**: Finding past cases with similar clinical presentations and contexts

These retrieval components combine rule-based and similarity-based approaches to provide comprehensive context for the final assessment.

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.outputs import LLMResult

class NodeTokenTracker(BaseCallbackHandler):
    """Tracks per-node LLM token usage via LangChain callbacks."""

    def __init__(self):
        self.current_node = "unknown"
        self.nodes: Dict[str, Dict[str, int]] = {}
        self.total = {"input": 0, "output": 0, "reasoning": 0, "total": 0}

    def set_node(self, node_name: str):
        self.current_node = node_name
        if node_name not in self.nodes:
            self.nodes[node_name] = {"input": 0, "output": 0, "reasoning": 0, "total": 0}

    def on_llm_end(self, response: LLMResult, **kwargs):
        try:
            usage = (response.llm_output or {}).get("token_usage", {})
            input_t = usage.get("prompt_tokens", 0) or 0
            output_t = usage.get("completion_tokens", 0) or 0
            total_t = usage.get("total_tokens", 0) or 0
            details = usage.get("completion_tokens_details") or {}
            reasoning_t = details.get("reasoning_tokens", 0) or 0

            node = self.nodes[self.current_node]
            node["input"] += input_t
            node["output"] += output_t
            node["reasoning"] += reasoning_t
            node["total"] += total_t

            self.total["input"] += input_t
            self.total["output"] += output_t
            self.total["reasoning"] += reasoning_t
            self.total["total"] += total_t
        except Exception as e:
            print(f"TokenTracker: could not extract usage for '{self.current_node}': {e}")

    def to_dict(self) -> Dict[str, Any]:
        return {
            "per_node": self.nodes,
            "total_input_tokens": self.total["input"],
            "total_output_tokens": self.total["output"],
            "total_reasoning_tokens": self.total["reasoning"],
            "total_tokens": self.total["total"],
        }

# Token usage tracking helper function
def get_token_usage(response) -> TokenUsageMetrics:
    """
    Extract token usage from LLM response if available.
    """
    try:
        # For OpenAI-compatible APIs, token usage is in response_metadata
        if hasattr(response, 'response_metadata') and 'token_usage' in response.response_metadata:
            usage = response.response_metadata['token_usage']
            return TokenUsageMetrics(
                prompt_tokens=usage.get('prompt_tokens', 0),
                completion_tokens=usage.get('completion_tokens', 0),
                total_tokens=usage.get('total_tokens', 0)
            )
    except Exception as e:
        print(f"Could not extract token usage: {e}")
    
    return TokenUsageMetrics()

# Retry logic for structured output nodes
def execute_structured_node_with_retry(node_func, state, node_name, max_retries=3):
    """
    Execute a node with retry logic for malformed structured responses.
    
    Args:
        node_func: The node function to execute
        state: Current graph state
        node_name: Name of the node (for logging)
        max_retries: Maximum number of retry attempts
        
    Returns:
        Updated state or state with failure flag
    """
    for attempt in range(max_retries):
        try:
            print(f"\n=== {node_name}: Attempt {attempt + 1}/{max_retries} ===")
            result = node_func(state)
            print(f"{node_name} succeeded on attempt {attempt + 1}")
            return result
        except (ValidationError, ValueError, KeyError, json.JSONDecodeError) as e:
            print(f"{node_name} attempt {attempt + 1} failed: {e}")
            if attempt == max_retries - 1:
                # Mark case as failed after all retries exhausted
                print(f"{node_name} FAILED after {max_retries} retries")
                failed_state = state.copy()
                failed_state["processing_failed"] = True
                failed_state["failure_reason"] = f"{node_name} failed after {max_retries} retries: {str(e)}"
                failed_state["failed_node"] = node_name
                return failed_state
        except Exception as e:
            print(f"{node_name} attempt {attempt + 1} failed with unexpected error: {e}")
            if attempt == max_retries - 1:
                failed_state = state.copy()
                failed_state["processing_failed"] = True
                failed_state["failure_reason"] = f"{node_name} failed with unexpected error: {str(e)}"
                failed_state["failed_node"] = node_name
                return failed_state
    
    return state

In [ ]:
# Define the node to process the case and select relevant sections with structured output
def guideline_retrieval_decision(state: GraphState, token_tracker=None) -> GraphState:
    """
    Guideline retrieval with structured output focusing on chief complaint and abnormal vitals.
    Uses native Pydantic structured output - no prompt engineering for JSON format needed.
    """

    # Extract key clinical factors from summary
    summary_text = state["summary_text"]

    # Keep the original focused prompt for guideline selection (no JSON format instructions)
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an emergency triage assistant. Your task is to select ONLY the 1-2 most critical guideline sections based on the chief complaint and abnormal vital signs.

        CRITICAL SELECTION RULES:
        1. Focus ONLY on the PRIMARY presenting complaint and documented ABNORMAL vital signs
        2. Select MAXIMUM 2 sections - prioritize life-threatening conditions first
        3. Do NOT select sections for normal findings or assumptions
        4. Consider abnormal vitals ONLY if they meet these thresholds:
           - BP: SBP > 180 or < 90 mmHg, DBP > 100 mmHg
           - HR: > 120 or < 50 (adults)
           - Temp: ≥ 38°C or < 35°C
           - Blood sugar: > 16 or < 3.9 mmol/l
           - SpO2: < 95% or requiring oxygen support
           - GCS: < 15

        SELECTION PRIORITY:
        1. Life-threatening symptoms (chest pain, SOB, altered consciousness)
        2. Abnormal vital signs meeting thresholds above
        3. Specific documented complaints
        4. If no specific sections apply, select none.
        """,
            ),
            (
                "user",
                """
        Clinical Summary: {summary_text}
        
        Available Guideline Sections:
        {guideline_section_metadata}
        
        Based on the clinical summary, which guideline sections are most relevant?
        """,
            ),
        ]
    )

    # Prepare input
    input_values = {
        "summary_text": summary_text,
        "guideline_section_metadata": "\n".join(
            [
                f"Section: {meta['section_title']}\nSummary: {meta['summary']}\n"
                for meta in state["guideline_section_metadata"]
            ]
        ),
    }

    structured_llm = llm.with_structured_output(GuidelineRetrievalOutput)
    chain = prompt | structured_llm
    _cb = {"callbacks": [token_tracker]} if token_tracker else {}
    MAX_RETRIES = 3
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = chain.invoke(input_values, config=_cb)
            break
        except Exception as e:
            last_error = e
            print(f"[guideline_retrieval_decision retry {attempt}/{MAX_RETRIES}] {type(e).__name__}: {e}")
            if attempt == MAX_RETRIES:
                raise last_error

    print(f"Selected sections: {response.selected_sections}")
    print(f"Reasoning: {response.reasoning}")

    # Update state with structured response
    updated_state = state.copy()
    updated_state["selected_guideline_sections"] = response.selected_sections
    updated_state["guideline_reasoning"] = response.reasoning
    
    return updated_state


# Define a node to retrieve the full content of selected sections
def retrieved_guideline_section_content(state: GraphState) -> GraphState:
    """
    Retrieve the full content of the selected sections.

    Args:
        state: Current state with selected section titles

    Returns:
        Updated state with retrieved section content
    """
    selected_guideline_sections = state["selected_guideline_sections"]
    full_guidelines = triage_guidelines  # We loaded this earlier

    # Find the full content for each selected section
    retrieved_guideline_content = []
    for section in full_guidelines:
        if section["section_title"] in selected_guideline_sections:
            retrieved_guideline_content.append(section)

    # Update the state
    updated_state = state.copy()
    updated_state["retrieved_guideline_content"] = retrieved_guideline_content
    return updated_state

In [ ]:
def attending_specialty_decision(state: GraphState, token_tracker=None) -> GraphState:
    """
    Predict attending specialty with native structured output.
    Uses dynamic schema creation based on AGENT_SPECIALTY_LIST variable.
    """

    # Extract case summary and specialty list
    summary_text = state["summary_text"]
    specialty_list = state["specialty_list"]  # This is AGENT_SPECIALTY_LIST

    # Create dynamic schema based on the actual AGENT_SPECIALTY_LIST
    SpecialtyPredictionOutput = create_specialty_prediction_schema(specialty_list)

    # Create prompt using AGENT_SPECIALTY_LIST variable
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an emergency-medicine specialist assistant that helps determine the most appropriate attending specialty for presenting cases.
        Given a medical case summary, analyze the information and determine:
        1. The primary attending specialty that should handle the case
        2. A possible secondary attending specialty (if applicable)
        
        You must select specialties from the following list only:
        
        {specialty_list}


        Some clinical presentation may have overlapping specialty coverage. For example:
           - Injury-related cases could be covered by "Surgery", "Orthopaedics" or "Neurosurgery"
           - Paediatric cases could be covered by "Paediatrics" and other specialties
           If overlapping specialty is possible, include in "secondary_specialty"

        """,
            ),
            (
                "user",
                """
        Clinical Summary: {summary_text}
        
        Based on this clinical presentation, which medical specialty should handle this case?
        """,
            ),
        ]
    )

    # Prepare input with current AGENT_SPECIALTY_LIST
    input_values = {
        "summary_text": summary_text,
        "specialty_list": ", ".join(specialty_list)
    }

    structured_llm = llm.with_structured_output(SpecialtyPredictionOutput)
    chain = prompt | structured_llm
    _cb = {"callbacks": [token_tracker]} if token_tracker else {}
    MAX_RETRIES = 3
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = chain.invoke(input_values, config=_cb)
            break
        except Exception as e:
            last_error = e
            print(f"[attending_specialty_decision retry {attempt}/{MAX_RETRIES}] {type(e).__name__}: {e}")
            if attempt == MAX_RETRIES:
                raise last_error

    primary = response.specialties[0]
    secondary = response.specialties[1] if len(response.specialties) > 1 else None

    print("============= Dynamic Structured Specialty Prediction =============")
    print(f"Primary Specialty: {primary}")
    if secondary:
        print(f"Secondary Specialty: {secondary}")
    print(f"Explanation: {response.explanation}")

    updated_state = state.copy()
    updated_state["selected_attending_specialty"] = {
        "primary_specialty": primary,
        "secondary_specialty": secondary,
        "explanation": response.explanation,
    }

    return updated_state

In [ ]:
def retrieve_past_case(
    state: GraphState,
    summary_vectordb: Chroma,
    k: int = k_cases,
    metadata_threshold: float = 1.0,
    similarity_threshold: float = 0.7,  # New stricter similarity threshold
) -> GraphState:
    """
    Retrieve similar past cases with stricter similarity thresholds and limited to top 3.

    Args:
        state: Current state
        summary_vectordb: Vector database for case summaries
        k: Number of cases to retrieve (reduced to 3)
        metadata_threshold: Threshold for metadata matching
        similarity_threshold: Minimum similarity score to include a case

    Returns:
        Updated state with filtered past cases
    """

    ## Step 1: Extract case metadata as filter
    metadata = {}

    # Extract age_group if available
    if (
        "Demographics" in state["case_json_full"]
        and "Age" in state["case_json_full"]["Demographics"]
    ):
        age_text = state["case_json_full"]["Demographics"]["Age"]
        metadata["age_group"] = "Unknown"

        if "years" in age_text.lower():
            try:
                age_years = int(age_text.lower().split("years")[0].strip())
                if age_years >= 18:
                    metadata["age_group"] = "Adult"
                else:
                    metadata["age_group"] = "Paediatric"
            except (ValueError, IndexError):
                pass
        elif "months" in age_text.lower() or "month" in age_text.lower():
            metadata["age_group"] = "Paediatric"

    print(f"Base metadata filters: {metadata}")

    ## Step 2: Add Attending Specialty as filter
    specialty_values = []

    if state["selected_attending_specialty"]["primary_specialty"]:
        primary_specialty = state["selected_attending_specialty"]["primary_specialty"]
        primary_raw_values = map_specialty_to_dataset_values(primary_specialty)
        specialty_values.extend(primary_raw_values)
        print(
            f"Using primary specialty: {primary_specialty} (raw values: {primary_raw_values})"
        )

    if state["selected_attending_specialty"]["secondary_specialty"]:
        secondary_specialty = state["selected_attending_specialty"][
            "secondary_specialty"
        ]
        secondary_raw_values = map_specialty_to_dataset_values(secondary_specialty)
        specialty_values.extend(secondary_raw_values)
        print(
            f"Using secondary specialty: {secondary_specialty} (raw values: {secondary_raw_values})"
        )

    specialty_values = list(set(specialty_values))
    print(f"Unique specialty values for filtering: {specialty_values}")

    ## Step 3: Apply metadata filtering
    all_filtered_ids = set()

    for specialty_value in specialty_values:
        current_filters = metadata.copy()
        current_filters["Attending Specialty"] = specialty_value

        print(
            f"Applying filters with specialty value '{specialty_value}': {current_filters}"
        )

        filtered_ids = filter_documents_by_metadata(
            summary_vectordb, current_filters, threshold=metadata_threshold
        )

        print(f"Found {len(filtered_ids)} cases with specialty '{specialty_value}'")
        all_filtered_ids.update(filtered_ids)

    filtered_ids = list(all_filtered_ids)
    print(f"Found {len(filtered_ids)} unique cases after filtering")

    if not filtered_ids:
        all_docs = summary_vectordb.get()
        filtered_ids = all_docs["ids"]
        print(f"Falling back to all {len(filtered_ids)} cases")

    ## Step 4: Perform vector similarity search
    summary_embedding = embeddings.embed_query(state["summary_text"])

    # Search with increased k to allow for filtering by similarity threshold
    search_k = min(k * 2, len(filtered_ids))  # Search more to filter by similarity

    summary_results = (
        summary_vectordb.similarity_search_by_vector_with_relevance_scores(
            embedding=summary_embedding,
            k=search_k,
            filter={"doc_id": {"$in": filtered_ids}},
        )
    )

    ## Step 5: Apply similarity threshold and limit results
    final_results = []
    for doc, score in summary_results:
        similarity = 1 - score  # Convert distance to similarity

        # Apply stricter similarity threshold
        if similarity < similarity_threshold:
            print(
                f"Filtering out case {doc.metadata['doc_id']} with similarity {similarity:.3f} < {similarity_threshold}"
            )
            continue

        # Stop when we have enough high-quality cases
        if len(final_results) >= k:
            break

        std_specialty = None
        if "Attending Specialty" in doc.metadata:
            dataset_specialty = doc.metadata["Attending Specialty"]
            std_specialty = get_standardized_specialty(dataset_specialty)

        final_results.append(
            {
                "doc_id": doc.metadata["doc_id"],
                "original_filename": doc.metadata.get("original_filename", ""),
                "similarity_score": similarity,
                "metadata": {
                    k: v
                    for k, v in doc.metadata.items()
                    if k not in ["doc_id", "original_filename"]
                },
                "standardized_specialty": std_specialty,
                "content": doc.page_content,
            }
        )

    print(
        f"========= Retrieved {len(final_results)} high-quality past cases (similarity >= {similarity_threshold}) ==============="
    )
    updated_state = state.copy()
    updated_state["retrieved_cases"] = final_results
    return updated_state

In [ ]:
def retrieve_case_content(
    state: GraphState, case_json_dir: str
) -> GraphState:
    """
    Retrieve simplified JSON content for each retrieved case (removes Clinical Presentation and Vitals).

    Args:
        state: Current state with retrieved_cases list
        case_json_dir: Directory containing the full JSON case files

    Returns:
        Updated state with simplified retrieved_cases_content list
    """

    # Initialize retrieved_cases_content list
    simplified_case_contents = []

    # Check if we have retrieved cases
    if not state.get("retrieved_cases"):
        print("No retrieved cases found in state")
        updated_state = state.copy()
        updated_state["retrieved_cases_content"] = []
        return updated_state

    print(
        f"Found {len(state['retrieved_cases'])} cases to retrieve simplified content for"
    )

    # Loop through each retrieved case
    for case in state["retrieved_cases"]:
        doc_id = case["doc_id"]
        json_filename = f"{doc_id}.json"
        json_path = os.path.join(case_json_dir, json_filename)

        try:
            # Attempt to read the JSON file
            with open(json_path, "r") as f:
                case_json = json.load(f)

            # Simplify the case content - remove Clinical Presentation and Vitals
            simplified_case = case_json.copy()

            # Remove the detailed sections
            if "Clinical Presentation" in simplified_case:
                simplified_case.pop("Clinical Presentation")
            if "Vitals and Observations" in simplified_case:
                simplified_case.pop("Vitals and Observations")

            # Keep essential information: Demographics, Case Disposition, Summary
            essential_case = {
                "Demographics": simplified_case.get("Demographics", {}),
                "Clinical Summary": simplified_case.get("_summary_text", ""),
                "Case Disposition": simplified_case.get("Case Disposition", {}),
            }

            # Add to our list with metadata for reference
            simplified_case_contents.append(
                {
                    "doc_id": doc_id,
                    "original_filename": case.get("original_filename", ""),
                    "similarity_score": case.get("similarity_score", 0.0),
                    "content": essential_case,
                }
            )
            print(f"Successfully loaded simplified content for {doc_id}")

        except FileNotFoundError:
            print(
                f"Warning: JSON file not found for doc_id: {doc_id} at path: {json_path}"
            )
            simplified_case_contents.append(
                {
                    "doc_id": doc_id,
                    "original_filename": case.get("original_filename", ""),
                    "similarity_score": case.get("similarity_score", 0.0),
                    "content": None,
                    "error": "File not found",
                }
            )

        except json.JSONDecodeError:
            print(f"Warning: Invalid JSON format in file for doc_id: {doc_id}")
            simplified_case_contents.append(
                {
                    "doc_id": doc_id,
                    "original_filename": case.get("original_filename", ""),
                    "similarity_score": case.get("similarity_score", 0.0),
                    "content": None,
                    "error": "Invalid JSON format",
                }
            )

    updated_state = state.copy()
    updated_state["retrieved_cases_content"] = simplified_case_contents

    print("=" * 80)
    print(f"Added {len(simplified_case_contents)} simplified case contents to state")

    return updated_state

### 3.3 Prediction node

The prediction node integrates all retrieved information to generate a structured triage assessment. It follows a stepwise approach that:
1. CLINICAL RISK ASSESSMENT (Based on Triage Definitions)
2. GUIDELINE-BASED ASSESSMENT (Medical Condition Specific)
3. REAL-WORLD FACTORS ANALYSIS (Past Case Patterns)
4. FINAL DECISION

In [ ]:
def final_triage_prediction(state: GraphState, token_tracker=None) -> GraphState:
    """
    Simplified 3-step triage prediction with native structured output.
    Uses original medical reasoning prompts with automatic Pydantic structuring.
    """

    # Keep the original comprehensive medical reasoning prompt
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """You are an emergency triage specialist. Use this 3-step approach to determine the appropriate triage category.

        TRIAGE CATEGORY DEFINITIONS:
        | Triage Category | Patient Conditions | Actions of Staff | Target Response Time (Time from registration to medical consultation) |
        |---------------|------------------|----------------|---------------------------------------------------------------------|
        | 1 Critical | • Suffer from a life-threatening condition(s) caused by a major event • With unstable vital signs requiring immediate resuscitation | • Direct patient to resuscitation room • Attend patient immediately by a team comprising medical and nursing staff | • Immediate • 100% of cases within the target response time |
        | 2 Emergency | • Suffer from a potentially life-threatening condition • Borderline vital signs but with potential risk of rapid deterioration • Require emergency treatment and immediate continuous close monitoring | • Direct patient to resuscitation room / treatment cubicle • Offer medical attention and immediate continuous close monitoring within 15 mins. | • < 15 mins • 95% of cases within the target response time |
        | 3 Urgent | • Suffer from a major condition with potential risk of deterioration • Stable vital signs | • Direct patient to cubicle | • < 30 mins • 90% of cases within the target response time |
        | 4 Semi-urgent | • Suffer from acute but stable condition(s) • Stable vital signs • Can afford to wait some time without serious complications | • Direct patient to cubicle / walk-in clinic | • < 120 mins |
        | 5 Non-urgent | • Suffer from minor and stable condition(s) (including acute and non-acute conditions) • Can afford to wait without deterioration • Stable vital signs | • Direct patient to walk-in clinic | Remarks: • Conditions can be treated in primary health care facilities • Should be based on clinical judgement only. Economic, social factors and availability of facilities should not be taken into consideration. |

        USE THIS 3-STEP APPROACH:

        USE THIS EXACT 3-STEP FORMAT:

        ### STEP 1: CLINICAL RISK ASSESSMENT (Based on Triage Definitions)
        **Focus:** Determine overall CLINICAL RISK and URGENCY based on triage category definitions
        **Patient Profile:** [Age, sex, key symptoms, abnormal vitals only]
        **Risk of Deterioration:** [Immediate/High/Moderate/Low - based on clinical instability]
        **Urgency Level:** [Life-threatening/Potentially life-threatening/Major/Acute/Minor]
        **Initial Triage Category (Clinical Risk):** [1/2/3/4/5]

        ### STEP 2: GUIDELINE-BASED ASSESSMENT (Medical Condition Specific)
        **Focus:** Apply specific medical guidelines for documented conditions
        **Most Relevant Guideline:** [Title of guideline, or "None specific"]
        **Specific Medical Condition:** [Condition addressed by guideline]
        **Guideline Criteria:** [Specific criteria from guideline]
        **Criteria Met:** [Yes/No with evidence]
        **Guideline-Recommended Category:** [1/2/3/4/5, or "Not specified"]

        ### STEP 3: REAL-WORLD FACTORS ANALYSIS (Past Case Patterns)
        **Focus:** Analyze factors OUTSIDE formal guidelines that influence real-world triage decisions
        
        **Case-by-Case Analysis:**
        #### Past Case 1:
        - **Demographics & Status:** [Age, ambulatory status, institutionalized status, referral status]
        - **Co-morbidities:** [Relevant past health conditions]
        - **Assigned Category:** [Category]
        - **Key Similarities to Current Case:** [List similarities]
        - **Key Differences from Current Case:** [List differences]
        - **Non-Guideline Factors:** [Age-related, mobility, social factors that may have influenced decision]

        [Repeat for each case]

        #### Pattern Recognition:
        - **Common Factors in Similar Cases:** [Factors appearing across multiple cases]
        - **Deviation Patterns from Guidelines:** [How similar cases deviate from strict guideline application]
        - **Implicit Decision Rules:** [Unwritten rules evident in past case decisions]
        - **Impact of Special Factors:** [How factors like age, ambulatory status, etc. affect categorization]

        ### FINAL DECISION
        Synthesize all three steps into final triage category with confidence assessment and full rationale to support each of your steps.
        
        CRITICAL INSTRUCTIONS:
        - Step 1: Focus purely on clinical urgency and risk of deterioration per triage definitions
        - Step 2: Apply medical guidelines for specific documented conditions only (always assign 1-5)
        - Step 3: Analyze ALL provided past cases individually - look for real-world factors beyond guidelines
        - Final: Explain which step(s) drove the decision and any adjustments made
        """,
            ),
            (
                "user",
                """
        Current Case:
        {case_summary}
        
        Relevant Guidelines:
        {guideline_content}
        
        Similar Past Cases:
        {past_cases_context}
        
        Please provide your 3-step triage assessment:
        """,
            ),
        ]
    )

    # Prepare simplified inputs (same as before)
    case_summary = f"""
    Clinical Summary: {state['summary_text']}
    
    Demographics: {state['case_json_full'].get('Demographics', {})}
    """

    # Simplified guideline content
    guideline_content = ""
    if state["retrieved_guideline_content"]:
        for section in state["retrieved_guideline_content"]:
            guideline_content += (
                f"GUIDELINE: {section['section_title']}\n{section['content']}...\n\n"
            )
    else:
        guideline_content = "No specific guidelines retrieved."

    # Simplified past cases context
    case_context = ""
    for i, case in enumerate(state["retrieved_cases_content"]):
        if case.get("content"):
            content = case["content"]
            case_context += f"Case {i+1}: {content.get('Demographics', {})}, "
            case_context += f"Summary: {content.get('Clinical Summary', '')}, "
            case_context += f"Category: {content.get('Case Disposition', {}).get('Triage Category', '')}\n\n"

    # Prepare input
    input_values = {
        "case_summary": case_summary,
        "guideline_content": guideline_content,
        "past_cases_context": case_context,
    }

    structured_llm = llm.with_structured_output(TriagePredictionOutput)
    chain = prompt | structured_llm
    _cb = {"callbacks": [token_tracker]} if token_tracker else {}
    MAX_RETRIES = 3
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = chain.invoke(input_values, config=_cb)
            break
        except Exception as e:
            last_error = e
            print(f"[final_triage_prediction retry {attempt}/{MAX_RETRIES}] {type(e).__name__}: {e}")
            if attempt == MAX_RETRIES:
                raise last_error

    print("======== NATIVE STRUCTURED FINAL TRIAGE ASSESSMENT ========")
    print(f"Step 1 (Clinical Risk): {response.step1_clinical_risk.category} - {response.step1_clinical_risk.confidence}")
    print(f"Step 2 (Guidelines): {response.step2_guidelines.category} - {response.step2_guidelines.confidence}")
    print(f"Step 3 (Real-world): {response.step3_realworld_factors.category} - {response.step3_realworld_factors.confidence}")
    print(f"FINAL DECISION: {response.final_decision.category} - {response.final_decision.confidence}")
    print("=" * 60)

    # Update state with structured response
    updated_state = state.copy()
    updated_state["final_assessment"] = response.model_dump()  # Convert Pydantic model to dict
    updated_state["final_category"] = response.final_decision.category
    updated_state["final_confidence"] = response.final_decision.confidence

    return updated_state

### Pipeline Assembly

The following functions assemble the individual nodes into a complete processing pipeline using LangGraph. The graph structure defines:
1. The execution order of processing steps
2. Data flow between components
3. State management throughout the pipeline

In [ ]:
def build_triage_graph(summary_vectordb: Chroma, case_json_dir: str):
    """
    Build the triage assessment graph.

    Args:
        summary_vectordb: Vector database for past cases
        case_json_dir: Directory containing the case JSON files

    Returns:
        The constructed triage graph
    """
    # Initialize the graph
    graph = StateGraph(GraphState)
    # Wrap retrieval function with parameters
    retrieve_past_case_with_params = (
        lambda state: retrieve_past_case(
            state=state,
            summary_vectordb=summary_vectordb,
            k=k_cases,
            metadata_threshold=1.0,
            similarity_threshold=0.7,  # Stricter similarity threshold
        )
    )
    # Wrap case content retrieval function
    retrieve_case_content_with_dir = (
        lambda state: retrieve_case_content(
            state=state, case_json_dir=case_json_dir
        )
    )
    
    # Wrap structured nodes with retry logic
    guideline_retrieval_with_retry = lambda state: execute_structured_node_with_retry(
        guideline_retrieval_decision, state, "guideline_retrieval_decision"
    )
    final_prediction_with_retry = lambda state: execute_structured_node_with_retry(
        final_triage_prediction, state, "final_triage_prediction"
    )
    specialty_prediction_with_retry = lambda state: execute_structured_node_with_retry(
        attending_specialty_decision, state, "attending_specialty_decision"
    )
    # Add nodes
    graph.add_node("process_case_summary", process_case_summary)
    graph.add_node(
        "guideline_retrieval", guideline_retrieval_decision
    )
    graph.add_node("guideline_retrieve_content", retrieved_guideline_section_content)
    graph.add_node("attending_specialty_decision", specialty_prediction_with_retry)
    graph.add_node(
        "retrieve_past_case", retrieve_past_case_with_params
    )
    graph.add_node(
        "retrieve_case_content", retrieve_case_content_with_dir
    )
    graph.add_node("final_predict_category", final_prediction_with_retry)
    # Define edges
    graph.add_edge("process_case_summary", "guideline_retrieval")
    graph.add_edge("guideline_retrieval", "guideline_retrieve_content")
    graph.add_edge("guideline_retrieve_content", "attending_specialty_decision")
    graph.add_edge("attending_specialty_decision", "retrieve_past_case")
    graph.add_edge("retrieve_past_case", "retrieve_case_content")
    graph.add_edge("retrieve_case_content", "final_predict_category")
    graph.add_edge("final_predict_category", END)
    # Set the entry point
    graph.set_entry_point("process_case_summary")
    # Compile the graph
    return graph.compile()

In [ ]:
# Initialize the triage pipeline
print("Initializing triage pipeline...")
triage_graph = build_triage_graph(
    summary_vectordb=summary_vectordb, case_json_dir=case_json_dir
)
print("Triage pipeline ready!")

triage_graph

## Step 4: Evaluation Framework


The following sections implement a testing framework to evaluate the system's performance on real emergency cases. This enables:
1. Validating the quality of guideline retrieval
2. Assessing the relevance of retrieved similar cases
3. Analyzing the accuracy and reasoning of final triage assessments

## Single Case Testing

This function allows testing the system on individual cases to examine the detailed processing steps and outputs.

In [ ]:
def process_case_file(file_path: str) -> Dict[str, Any]:
    """
    Process a case file using the triage assessment pipeline.

    Args:
        file_path (str): Path to the case file (json)

    Returns:
        Dict[str, Any]: Complete state from the triage graph execution
    """
    # Load and format the case
    case_data = load_case_from_json(file_path)

    # Print the case content for inspection
    print("Case Content:")
    print(f"Age: {case_data.get('Demographics', {}).get('Age', 'Unknown')}")
    print(f"Sex: {case_data.get('Demographics', {}).get('Sex', 'Unknown')}")
    print(
        f"Chief Complaint: {case_data.get('Clinical Presentation', {}).get('Chief complaint', 'Unknown')[:100]}..."
    )
    print("-" * 80)

    # Run the triage system
    initial_state = GraphState(
        case_json_full=case_data,
        guideline_section_metadata=guideline_section_metadata,
        specialty_list=AGENT_SPECIALTY_LIST,
    )

    # Return the complete result state
    return triage_graph.invoke(
        initial_state,
        config={"retrieve_case_content": {"case_json_dir": case_json_dir}},
    )

In [ ]:
# addition of time logging 
def process_case_file_with_timing(file_path: str) -> Tuple[Dict[str, Any], Dict[str, float]]:
    """
    Process a case file using the triage assessment pipeline with detailed timing.

    Args:
        file_path (str): Path to the case file (json)

    Returns:
        Tuple[Dict[str, Any], Dict[str, float]]: Complete state and timing metrics
    """
    # Initialize timing dictionary
    timing_metrics = {}
    
    # Record start time
    start_time = time.time()
    
    # Load and format the case
    case_data = load_case_from_json(file_path)

    # Print the case content for inspection
    print("Case Content:")
    print(f"Age: {case_data.get('Demographics', {}).get('Age', 'Unknown')}")

    # Initialize per-node token tracker
    token_tracker = NodeTokenTracker()
    print(f"Sex: {case_data.get('Demographics', {}).get('Sex', 'Unknown')}")
    print(
        f"Chief Complaint: {case_data.get('Clinical Presentation', {}).get('Chief complaint', 'Unknown')[:100]}..."
    )
    print("-" * 80)

    # Prepare initial state
    initial_state = GraphState(
        case_json_full=case_data,
        guideline_section_metadata=guideline_section_metadata,
        specialty_list=AGENT_SPECIALTY_LIST,
    )

    # Execute each node individually with timing
    
    # 1. Preprocessing Node
    preprocessing_start = time.time()
    token_tracker.set_node("summary")
    state = process_case_summary(initial_state, token_tracker=token_tracker)
    preprocessing_end = time.time()
    timing_metrics["preprocessing_time"] = preprocessing_end - preprocessing_start
    
    # 2. Guideline Retrieval
    guideline_start = time.time()
    token_tracker.set_node("guideline_retrieval")
    state = guideline_retrieval_decision(state, token_tracker=token_tracker)
    state = retrieved_guideline_section_content(state)
    guideline_end = time.time()
    timing_metrics["guideline_retrieval_time"] = guideline_end - guideline_start
    
    # 3. Specialty Decision (part of retrieval)
    specialty_start = time.time()
    token_tracker.set_node("specialty_decision")
    state = attending_specialty_decision(state, token_tracker=token_tracker)
    specialty_end = time.time()
    timing_metrics["specialty_decision_time"] = specialty_end - specialty_start
    
    # 4. Past Case Retrieval
    past_case_start = time.time()
    state = retrieve_past_case(
        state=state,
        summary_vectordb=summary_vectordb,
        k=k_cases,
        metadata_threshold=1.0,
        similarity_threshold=0.7,
    )
    state = retrieve_case_content(
        state=state, case_json_dir=case_json_dir
    )
    past_case_end = time.time()
    timing_metrics["past_case_retrieval_time"] = past_case_end - past_case_start
    
    # 5. Prediction Node
    prediction_start = time.time()
    token_tracker.set_node("prediction")
    state = final_triage_prediction(state, token_tracker=token_tracker)
    prediction_end = time.time()
    timing_metrics["prediction_time"] = prediction_end - prediction_start
    
    # Calculate total time
    end_time = time.time()
    timing_metrics["total_processing_time"] = end_time - start_time
    
    # Calculate component breakdown
    timing_metrics["total_retrieval_time"] = (
        timing_metrics["guideline_retrieval_time"] + 
        timing_metrics["specialty_decision_time"] + 
        timing_metrics["past_case_retrieval_time"]
    )
    
    # Add percentage breakdown
    total_time = timing_metrics["total_processing_time"]
    if total_time > 0:
        timing_metrics["preprocessing_percentage"] = (timing_metrics["preprocessing_time"] / total_time) * 100
        timing_metrics["guideline_retrieval_percentage"] = (timing_metrics["guideline_retrieval_time"] / total_time) * 100
        timing_metrics["specialty_decision_percentage"] = (timing_metrics["specialty_decision_time"] / total_time) * 100
        timing_metrics["past_case_retrieval_percentage"] = (timing_metrics["past_case_retrieval_time"] / total_time) * 100
        timing_metrics["prediction_percentage"] = (timing_metrics["prediction_time"] / total_time) * 100
        timing_metrics["total_retrieval_percentage"] = (timing_metrics["total_retrieval_time"] / total_time) * 100


    # Add token usage to timing metrics
    timing_metrics["token_usage"] = token_tracker.to_dict()

    # Check if processing failed
    if state.get("processing_failed", False):
        print(f"Processing failed: {state.get('failure_reason', 'Unknown error')}")
        # Return the failed state with timing info
        return state, timing_metrics

    return state, timing_metrics


In [ ]:
# process_case_file(
#     file_path="PATH_TO_TEST_JSON"
# )

## Batch Evaluation

This section implements a framework for processing multiple cases and saving the results for systematic evaluation. The outputs include:
1. Case summaries
2. Retrieved guideline sections
3. Similar past cases
4. Complete triage assessments with reasoning

These structured outputs can be analyzed to evaluate the system's performance across different case types and triage categories.

In [ ]:
def batch_process_directory(
    input_directory: str,
    output_directory: str,
    max_files: int = None,
    verbose: bool = True,
) -> int:
    """
    Process all JSON files and save simplified results.
    """
    # Create output directory
    os.makedirs(output_directory, exist_ok=True)

    # Find all JSON files
    json_files = []
    for root, _, files in os.walk(input_directory):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))

    if max_files:
        json_files = json_files[:max_files]

    print(f"Found {len(json_files)} JSON files to process")

    # Track statistics
    processed_count = 0
    skipped_count = 0
    error_count = 0

    # Process each file
    for file_path in tqdm(json_files, desc="Processing cases (guideline + past case)"):
        try:
            # Get relative path for output structure
            rel_path = os.path.relpath(file_path, input_directory)
            file_name = os.path.basename(file_path)
            base_name = os.path.splitext(file_name)[0]

            # Create output subdirectory
            rel_dir = os.path.dirname(rel_path)
            output_subdir = os.path.join(output_directory, rel_dir)
            os.makedirs(output_subdir, exist_ok=True)

            # Check if output file already exists
            output_file_path = os.path.join(
                output_subdir, f"{base_name}_pastcase_guideline_output.json"
            )
            if os.path.exists(output_file_path):
                if verbose:
                    print(
                        f"\nSkipping {rel_path} - output file already exists"
                    )
                skipped_count += 1
                continue

            if verbose:
                print(f"\nProcessing: {rel_path}")

            # Process the case
            state = process_case_file(file_path)

            # Save simplified output data
            output_data = {
                "summary_text": state.get("summary_text"),
                "case_json_full": state.get("case_json_full"),
                "selected_guideline_sections": state.get(
                    "selected_guideline_sections", []
                ),
                "guideline_count": len(state.get("selected_guideline_sections", [])),
                "retrieved_cases_simplified": [
                    {
                        "doc_id": case.get("doc_id"),
                        "similarity_score": case.get("similarity_score"),
                        "content": case.get("content"),
                    }
                    for case in state.get("retrieved_cases_content", [])
                ],
                "past_cases_count": len(state.get("retrieved_cases_content", [])),
                "final_assessment": state.get("final_assessment"),
                "final_category": state.get("final_category"),
                "final_confidence": state.get("final_confidence"),
                "processing_failed": state.get("processing_failed", False),
                "failure_reason": state.get("failure_reason"),
            }

            # Save the output
            with open(output_file_path, "w", encoding="utf-8") as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)

            processed_count += 1

            if verbose:
                print(f"Saved output to: {output_file_path}")
                print(
                    f"Guidelines retrieved: {len(state.get('selected_guideline_sections', []))}"
                )
                print(
                    f"Past cases retrieved: {len(state.get('retrieved_cases_content', []))}"
                )

        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            print(traceback.format_exc())
            error_count += 1

    # Print summary
    print("\nProcessing Complete")
    print(f"Total files found: {len(json_files)}")
    print(f"Files processed: {processed_count}")
    print(f"Files skipped (already exist): {skipped_count}")
    print(f"Files with errors: {error_count}")

    return processed_count

In [ ]:
# addition of metrics logging 
def batch_process_directory_with_metrics(
    input_directory: str,
    output_directory: str,
    max_files: int = None,
    verbose: bool = True,
) -> int:
    """
    Process all JSON files and save results with detailed timing metrics.
    
    Args:
        input_directory (str): Directory containing input JSON files
        output_directory (str): Directory to save results and timing metrics
        max_files (int, optional): Maximum number of files to process
        verbose (bool): Whether to print detailed progress information
        
    Returns:
        int: Number of files successfully processed
    """
    # Create output directories
    os.makedirs(output_directory, exist_ok=True)
    metrics_dir = os.path.join(output_directory, "case_metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    # Find all JSON files
    json_files = []
    for root, _, files in os.walk(input_directory):
        for file in files:
            if file.endswith(".json"):
                json_files.append(os.path.join(root, file))

    if max_files:
        json_files = json_files[:max_files]

    print(f"Found {len(json_files)} JSON files to process (with metrics)")

    # Track statistics
    processed_count = 0
    skipped_count = 0
    error_count = 0
    failed_count = 0
    
    # Aggregate timing statistics
    aggregate_timing = {
        "total_files": 0,
        "successful_files": 0,
        "avg_total_time": 0,
        "avg_preprocessing_time": 0,
        "avg_guideline_retrieval_time": 0,
        "avg_specialty_decision_time": 0,
        "avg_past_case_retrieval_time": 0,
        "avg_prediction_time": 0,
        "avg_total_retrieval_time": 0,
        "min_total_time": float('inf'),
        "max_total_time": 0,
        "total_processing_time_all_files": 0
    }
    
    all_timing_data = []

    aggregate_tokens = {
        "total_input_tokens_all": 0,
        "total_output_tokens_all": 0,
        "total_reasoning_tokens_all": 0,
        "total_tokens_all": 0,
        "min_total_tokens": float('inf'),
        "max_total_tokens": 0,
        "avg_total_tokens": 0,
        "avg_input_tokens": 0,
        "avg_output_tokens": 0,
        "avg_reasoning_tokens": 0,
        "avg_summary_tokens": 0,
        "avg_guideline_retrieval_tokens": 0,
        "avg_specialty_decision_tokens": 0,
        "avg_prediction_tokens": 0,
    }

    # Process each file
    for file_path in tqdm(json_files, desc="Processing cases (guideline + past case, with metrics)"):
        try:
            # Get relative path for output structure
            rel_path = os.path.relpath(file_path, input_directory)
            file_name = os.path.basename(file_path)
            base_name = os.path.splitext(file_name)[0]

            # Create output subdirectory
            rel_dir = os.path.dirname(rel_path)
            output_subdir = os.path.join(output_directory, rel_dir)
            os.makedirs(output_subdir, exist_ok=True)

            # Check if output files already exist
            output_file_path = os.path.join(
                output_subdir, f"{base_name}_pastcase_guideline_output.json"
            )
            timing_file_path = os.path.join(
                metrics_dir, f"{base_name}_case_metrics.json"
            )
            
            if os.path.exists(output_file_path) and os.path.exists(timing_file_path):
                if verbose:
                    print(
                        f"\nSkipping {rel_path} - output and timing files already exist"
                    )
                skipped_count += 1
                continue

            if verbose:
                print(f"\nProcessing: {rel_path}")

            # Process the case with timing
            batch_start_time = time.time()
            state, timing_metrics = process_case_file_with_timing(file_path)
            batch_end_time = time.time()
            

            # Check if case processing failed
            if state.get("processing_failed", False):
                print(f"Case processing failed: {timing_metrics.get('failure_reason', 'Unknown')}")
                failed_count += 1
                # Log the failure but don't save output files
                failure_log = {
                    "file_path": rel_path,
                    "file_name": file_name,
                    "case_id": base_name,
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()),
                    "failure_reason": state.get("failure_reason", "Unknown"),
                    "failed_node": state.get("failed_node", "Unknown"),
                    "timing_metrics": timing_metrics
                }
                
                failure_log_path = os.path.join(metrics_dir, f"{base_name}_FAILED.json")
                with open(failure_log_path, "w", encoding="utf-8") as f:
                    json.dump(failure_log, f, indent=2, ensure_ascii=False)
                
                continue  # Skip to next file, don't save regular output

            # Add batch processing overhead time
            timing_metrics["batch_processing_overhead"] = (batch_end_time - batch_start_time) - timing_metrics["total_processing_time"]
            timing_metrics["batch_total_time"] = batch_end_time - batch_start_time

            # Save timing metrics to separate file
            timing_data = {
                "file_path": rel_path,
                "file_name": file_name,
                "case_id": base_name,
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()),
                "timing_metrics": timing_metrics,
                "case_metadata": {
                    "age": state.get("case_json_full", {}).get("Demographics", {}).get("Age", "Unknown"),
                    "sex": state.get("case_json_full", {}).get("Demographics", {}).get("Sex", "Unknown"),
                    "guidelines_retrieved": len(state.get("selected_guideline_sections", [])),
                    "past_cases_retrieved": len(state.get("retrieved_cases_content", [])),
                }
            }
            
            with open(timing_file_path, "w", encoding="utf-8") as f:
                json.dump(timing_data, f, indent=2, ensure_ascii=False)

            # Save main output data
            output_data = {
                "summary_text": state.get("summary_text"),
                "case_json_full": state.get("case_json_full"),
                "selected_guideline_sections": state.get(
                    "selected_guideline_sections", []
                ),
                "guideline_count": len(state.get("selected_guideline_sections", [])),
                "retrieved_cases_simplified": [
                    {
                        "doc_id": case.get("doc_id"),
                        "similarity_score": case.get("similarity_score"),
                        "content": case.get("content"),
                    }
                    for case in state.get("retrieved_cases_content", [])
                ],
                "past_cases_count": len(state.get("retrieved_cases_content", [])),
                "final_assessment": state.get("final_assessment"),
                "final_category": state.get("final_category"),
                "final_confidence": state.get("final_confidence"),
                "processing_failed": state.get("processing_failed", False),
                "failure_reason": state.get("failure_reason"),
                "timing_summary": {
                    "total_time": timing_metrics["total_processing_time"],
                    "preprocessing_time": timing_metrics["preprocessing_time"],
                    "retrieval_time": timing_metrics["total_retrieval_time"],
                    "prediction_time": timing_metrics["prediction_time"],
                }
            }

            with open(output_file_path, "w", encoding="utf-8") as f:
                json.dump(output_data, f, indent=2, ensure_ascii=False)

            # Update aggregate statistics
            all_timing_data.append(timing_metrics)
            aggregate_timing["successful_files"] += 1
            aggregate_timing["total_processing_time_all_files"] += timing_metrics["total_processing_time"]
            aggregate_timing["min_total_time"] = min(aggregate_timing["min_total_time"], timing_metrics["total_processing_time"])
            aggregate_timing["max_total_time"] = max(aggregate_timing["max_total_time"], timing_metrics["total_processing_time"])

            # Accumulate token usage
            _tok = timing_metrics.get("token_usage", {})
            _pn = _tok.get("per_node", {})
            aggregate_tokens["total_input_tokens_all"] += _tok.get("total_input_tokens", 0)
            aggregate_tokens["total_output_tokens_all"] += _tok.get("total_output_tokens", 0)
            aggregate_tokens["total_reasoning_tokens_all"] += _tok.get("total_reasoning_tokens", 0)
            aggregate_tokens["total_tokens_all"] += _tok.get("total_tokens", 0)
            aggregate_tokens["min_total_tokens"] = min(aggregate_tokens["min_total_tokens"], _tok.get("total_tokens", 0))
            aggregate_tokens["max_total_tokens"] = max(aggregate_tokens["max_total_tokens"], _tok.get("total_tokens", 0))
            aggregate_tokens["avg_summary_tokens"] += _pn.get("summary", {}).get("total", 0)
            aggregate_tokens["avg_guideline_retrieval_tokens"] += _pn.get("guideline_retrieval", {}).get("total", 0)
            aggregate_tokens["avg_specialty_decision_tokens"] += _pn.get("specialty_decision", {}).get("total", 0)
            aggregate_tokens["avg_prediction_tokens"] += _pn.get("prediction", {}).get("total", 0)

            processed_count += 1

            if verbose:
                print(f"Saved output to: {output_file_path}")
                print(f"Saved timing metrics to: {timing_file_path}")
                print(f"Total processing time: {timing_metrics['total_processing_time']:.3f}s")
                print(f"  - Preprocessing: {timing_metrics['preprocessing_time']:.3f}s ({timing_metrics['preprocessing_percentage']:.1f}%)")
                print(f"  - Guideline retrieval: {timing_metrics['guideline_retrieval_time']:.3f}s ({timing_metrics['guideline_retrieval_percentage']:.1f}%)")
                print(f"  - Past case retrieval: {timing_metrics['past_case_retrieval_time']:.3f}s ({timing_metrics['past_case_retrieval_percentage']:.1f}%)")
                print(f"  - Prediction: {timing_metrics['prediction_time']:.3f}s ({timing_metrics['prediction_percentage']:.1f}%)")
                _tok = timing_metrics.get("token_usage", {})
                print(f"  Token usage: {_tok.get('total_tokens', 0)} tokens  (in: {_tok.get('total_input_tokens', 0)}, out: {_tok.get('total_output_tokens', 0)})")

        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            print(traceback.format_exc())
            error_count += 1

    # Calculate and save aggregate timing statistics
    if aggregate_timing["successful_files"] > 0:
        # Calculate averages
        for timing_data in all_timing_data:
            for key in ["preprocessing_time", "guideline_retrieval_time", "specialty_decision_time", 
                       "past_case_retrieval_time", "prediction_time", "total_retrieval_time", "total_processing_time"]:
                if f"avg_{key}" not in aggregate_timing:
                    aggregate_timing[f"avg_{key}"] = 0
                aggregate_timing[f"avg_{key}"] += timing_data[key]
        
        # Divide by count to get averages
        for key in list(aggregate_timing.keys()):
            if key.startswith("avg_"):
                aggregate_timing[key] /= aggregate_timing["successful_files"]
    
    aggregate_timing["total_files"] = len(json_files)

    # Compute token averages
    _n = aggregate_timing["successful_files"]
    if _n > 0:
        aggregate_tokens["avg_total_tokens"] = aggregate_tokens["total_tokens_all"] / _n
        aggregate_tokens["avg_input_tokens"] = aggregate_tokens["total_input_tokens_all"] / _n
        aggregate_tokens["avg_output_tokens"] = aggregate_tokens["total_output_tokens_all"] / _n
        aggregate_tokens["avg_reasoning_tokens"] = aggregate_tokens["total_reasoning_tokens_all"] / _n
        for _k in ["avg_summary_tokens", "avg_guideline_retrieval_tokens",
                   "avg_specialty_decision_tokens", "avg_prediction_tokens"]:
            aggregate_tokens[_k] /= _n
        if aggregate_tokens["min_total_tokens"] == float('inf'):
            aggregate_tokens["min_total_tokens"] = 0
    
    # Save aggregate timing report
    aggregate_report = {
        "batch_summary": {
            "total_files_found": len(json_files),
            "files_processed": processed_count,
            "files_skipped": skipped_count,
            "files_with_errors": error_count,
            "processing_date": time.strftime("%Y-%m-%d %H:%M:%S", time.localtime()),
        },
        "timing_statistics": aggregate_timing,
        "token_statistics": aggregate_tokens,
        "performance_insights": {
            "avg_files_per_minute": (aggregate_timing["successful_files"] / (aggregate_timing["total_processing_time_all_files"] / 60)) if aggregate_timing["total_processing_time_all_files"] > 0 else 0,
            "fastest_case_time": aggregate_timing["min_total_time"],
            "slowest_case_time": aggregate_timing["max_total_time"],
            "time_range": aggregate_timing["max_total_time"] - aggregate_timing["min_total_time"],
        }
    }
    
    aggregate_report_path = os.path.join(metrics_dir, "aggregate_metrics_report.json")
    with open(aggregate_report_path, "w", encoding="utf-8") as f:
        json.dump(aggregate_report, f, indent=2, ensure_ascii=False)

    # Print summary
    print("\n" + "="*80)
    print("PROCESSING WITH TIMING COMPLETE")
    print("="*80)
    print(f"Total files found: {len(json_files)}")
    print(f"Files processed: {processed_count}")
    print(f"Files skipped (already exist): {skipped_count}")
    print(f"Files with errors: {error_count}")
    print("\nTIMING SUMMARY:")
    if aggregate_timing["successful_files"] > 0:
        print(f"Average total processing time per case: {aggregate_timing['avg_total_processing_time']:.3f}s")
        print(f"  - Average preprocessing time: {aggregate_timing['avg_preprocessing_time']:.3f}s")
        print(f"  - Average guideline retrieval time: {aggregate_timing['avg_guideline_retrieval_time']:.3f}s")
        print(f"  - Average past case retrieval time: {aggregate_timing['avg_past_case_retrieval_time']:.3f}s")
        print(f"  - Average prediction time: {aggregate_timing['avg_prediction_time']:.3f}s")
        print(f"Fastest case: {aggregate_timing['min_total_time']:.3f}s")
        print(f"Slowest case: {aggregate_timing['max_total_time']:.3f}s")
        print(f"Total processing time for all cases: {aggregate_timing['total_processing_time_all_files']:.2f}s")
        print(f"Average processing rate: {aggregate_report['performance_insights']['avg_files_per_minute']:.1f} files/minute")
    print(f"\nTiming metrics saved to: {metrics_dir}")
    print(f"Aggregate timing report saved to: {aggregate_report_path}")

    return processed_count

In [ ]:
batch_process_directory_with_metrics(
    input_directory="PATH_TO_INPUT_JSON_DIR",
    output_directory="PATH_TO_OUTPUT_DIR",
    max_files=None,
    verbose=True,
)